# Training Log Loss Trend

解析 `train_ctc.py` 的训练日志，绘制 `loss`、`loss_ctc`、`loss_decoder` 和学习率趋势，并标记 CTC nan-grad warning 出现的位置。

In [ ]:
from pathlib import Path
from collections import deque
import re

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True

# 在 spanish_sensevoice_asr/ 目录运行 notebook 时，默认日志路径如下。
# 如果你的日志文件名不同，改这里即可。
LOG_PATH = Path("logs/train_2gpu_dynamic_lfr.log")
TAIL_LINES = 500

if not LOG_PATH.exists():
    alt = Path("../logs/train_2gpu_dynamic_lfr.log")
    if alt.exists():
        LOG_PATH = alt

print(LOG_PATH.resolve())
print(f"tail lines: {TAIL_LINES}")

In [ ]:
train_pattern = re.compile(
    r"batch=(?P<batch>\d+)\s+"
    r"step=(?P<step>\d+)\s+"
    r"lr=(?P<lr>[0-9.eE+-]+)\s+"
    r"loss=(?P<loss>[0-9.eE+-]+)\s+"
    r"loss_ctc=(?P<loss_ctc>[0-9.eE+-]+)\s+"
    r"loss_decoder=(?P<loss_decoder>[0-9.eE+-]+)"
)
warning_pattern = re.compile(r"(?P<count>\d+)/(?P<total>\d+) samples got nan grad")

rows = []
warnings = []
last_step = None
last_batch = None

with LOG_PATH.open("r", encoding="utf-8", errors="replace") as f:
    lines = deque(f, maxlen=TAIL_LINES)

for line_no, line in enumerate(lines, start=1):
    match = train_pattern.search(line)
    if match:
        item = match.groupdict()
        row = {
            "tail_line_no": line_no,
            "batch": int(item["batch"]),
            "step": int(item["step"]),
            "lr": float(item["lr"]),
            "loss": float(item["loss"]),
            "loss_ctc": float(item["loss_ctc"]),
            "loss_decoder": float(item["loss_decoder"]),
        }
        rows.append(row)
        last_step = row["step"]
        last_batch = row["batch"]
        continue

    warning = warning_pattern.search(line)
    if warning:
        warnings.append({
            "tail_line_no": line_no,
            "step": last_step,
            "batch": last_batch,
            "nan_count": int(warning.group("count")),
            "batch_total": int(warning.group("total")),
            "raw": line.strip(),
        })

df = pd.DataFrame(rows)
warn_df = pd.DataFrame(warnings)

print(f"parsed tail lines: {len(lines)}")
print(f"parsed training points: {len(df)}")
print(f"parsed ctc warnings: {len(warn_df)}")
display(df.tail())
display(warn_df.tail())

In [ ]:
if df.empty:
    raise ValueError(f"No training lines were parsed from {LOG_PATH}")

# 日志一般每 100 个 batch 打一次，这里 rolling_window=20 约等于最近 2000 batch 的平滑趋势。
rolling_window = 20
plot_df = df.copy()
for col in ["loss", "loss_ctc", "loss_decoder"]:
    plot_df[f"{col}_ma"] = plot_df[col].rolling(rolling_window, min_periods=1).mean()

fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)

series = [
    ("loss", "loss_ma", "Average logged loss"),
    ("loss_ctc", "loss_ctc_ma", "CTC loss"),
    ("loss_decoder", "loss_decoder_ma", "Decoder loss"),
]

for ax, (raw_col, ma_col, title) in zip(axes, series):
    ax.plot(plot_df["step"], plot_df[raw_col], alpha=0.35, linewidth=1, label=raw_col)
    ax.plot(plot_df["step"], plot_df[ma_col], linewidth=2, label=f"{raw_col} rolling mean")
    if not warn_df.empty:
        for step in warn_df["step"].dropna().astype(int):
            ax.axvline(step, color="tab:red", alpha=0.12, linewidth=1)
    ax.set_title(title)
    ax.set_ylabel("loss")
    ax.legend(loc="best")

axes[-1].set_xlabel("step")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(plot_df["step"], plot_df["loss_ma"], label="loss rolling mean", color="tab:blue")
ax1.set_xlabel("step")
ax1.set_ylabel("loss", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(plot_df["step"], plot_df["lr"], label="lr", color="tab:orange", alpha=0.85)
ax2.set_ylabel("learning rate", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

if not warn_df.empty:
    for step in warn_df["step"].dropna().astype(int):
        ax1.axvline(step, color="tab:red", alpha=0.15, linewidth=1)

fig.suptitle("Loss trend vs learning rate")
fig.tight_layout()
plt.show()

In [ ]:
summary = df[["loss", "loss_ctc", "loss_decoder", "lr"]].describe().T
summary["first"] = df[["loss", "loss_ctc", "loss_decoder", "lr"]].iloc[0]
summary["last"] = df[["loss", "loss_ctc", "loss_decoder", "lr"]].iloc[-1]
summary["delta_last_first"] = summary["last"] - summary["first"]
display(summary)

if not warn_df.empty:
    warn_by_step = warn_df.dropna(subset=["step"]).copy()
    warn_by_step["step"] = warn_by_step["step"].astype(int)
    display(warn_by_step[["step", "batch", "nan_count", "batch_total", "raw"]].tail(50))
else:
    print("No CTC nan-grad warnings found.")